# Heart disease: exploration, model comparison and export

Adapted from the author's `heart_disease (2).ipynb`. This repository copy preserves the exploration and five-model comparison while correcting preprocessing and portable file paths. Original notebook files are unchanged.

This predicts the dataset's heart disease class, not stroke risk. Obtain the source CSV as described in `data/README.md`. The notebook's outputs are cleared for publication.

In [ ]:
from pathlib import Path  # Use portable project-relative paths.
import subprocess  # Run the shared reproducible export script.
import sys  # Use this notebook's Python interpreter.
import numpy as np  # Represent missing readings.
import pandas as pd  # Load and encode data.
import matplotlib.pyplot as plt  # Display exploration charts.
import seaborn as sns  # Plot feature distributions and relationships.
from sklearn.impute import SimpleImputer  # Learn means from training rows only.
from sklearn.model_selection import train_test_split  # Create held-out evaluation data.
from sklearn.preprocessing import StandardScaler  # Learn scaling from training rows only.
from sklearn.metrics import accuracy_score, f1_score, classification_report  # Evaluate model predictions.
from sklearn.linear_model import LogisticRegression  # Compare logistic regression.
from sklearn.naive_bayes import GaussianNB  # Compare Gaussian Naive Bayes.
from sklearn.tree import DecisionTreeClassifier  # Compare a decision tree.
from sklearn.svm import SVC  # Compare a support-vector classifier.
from sklearn.neighbors import KNeighborsClassifier  # Compare the deployed KNN classifier.
ROOT = Path.cwd().resolve()  # Start from the kernel's current directory.
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT  # Support launching from either project root or notebooks.
DATA_PATH = ROOT / "data" / "heart.csv"  # Use the local, ignored source dataset.
df = pd.read_csv(DATA_PATH)  # Load raw data before preprocessing.
df.head()  # Inspect the feature names and example values.


## Explore the original data

Zero cholesterol and resting blood pressure readings are treated as missing during training. Decimal values such as `Oldpeak` are retained.

In [ ]:
df.info()  # Inspect column types and missing values.
df.describe()  # Summarize numeric feature ranges.


In [ ]:
print("Duplicate rows:", df.duplicated().sum())  # Check duplicate records.
print(df.isna().sum())  # Check explicit missing values.
df["HeartDisease"].value_counts().sort_index().plot(kind="bar", title="Target classes")  # Inspect class balance.
plt.show()  # Display the target chart.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))  # Create four distribution panels.
for column, axis in zip(["Age", "RestingBP", "Cholesterol", "MaxHR"], axes.flat):  # Explore the original numeric features.
    sns.histplot(data=df, x=column, kde=True, ax=axis)  # Plot each distribution.
fig.tight_layout()  # Keep labels readable.
plt.show()  # Display the distributions.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))  # Create category comparison panels.
for column, axis in zip(["Sex", "ChestPainType", "FastingBS"], axes):  # Compare original categorical features.
    sns.countplot(data=df, x=column, hue="HeartDisease", ax=axis)  # Show counts by target class.
fig.tight_layout()  # Avoid overlapping chart labels.
plt.show()  # Display the category comparisons.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # Compare numeric features against the target.
sns.boxplot(data=df, x="HeartDisease", y="Cholesterol", ax=axes[0])  # Compare cholesterol distributions.
sns.violinplot(data=df, x="HeartDisease", y="Age", ax=axes[1])  # Compare age distributions.
fig.tight_layout()  # Space the charts clearly.
plt.show()  # Display both comparisons.
plt.figure(figsize=(8, 6))  # Create a readable correlation chart.
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f")  # Explore numeric correlations.
plt.tight_layout()  # Keep axis labels visible.
plt.show()  # Display the heatmap.


## Split, impute and scale

Split before learning any means or scaling statistics. Encode categories into the dataset's feature schema, keep the target separate, and reuse fitted preprocessing on the test subset. This matches `train.py`.

In [ ]:
cleaned = df.copy()  # Keep the exploratory raw data unchanged.
missing_columns = ["Cholesterol", "RestingBP"]  # Identify invalid zero readings.
cleaned[missing_columns] = cleaned[missing_columns].replace(0, np.nan)  # Mark zeros as missing without learning statistics.
X = pd.get_dummies(cleaned.drop(columns="HeartDisease"), drop_first=True, dtype=int)  # Encode categories while preserving decimals.
Y = cleaned["HeartDisease"]  # Select the target with square brackets.
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.20, random_state=42, stratify=Y)  # Preserve class proportions in a reproducible split.
imputer = SimpleImputer(strategy="mean").set_output(transform="pandas")  # Preserve feature names.
X_train_filled = imputer.fit_transform(X_train)  # Learn missing-value means from training rows.
X_test_filled = imputer.transform(X_test)  # Apply the same means to held-out rows.
scaler = StandardScaler()  # Create one training scaler.
X_train_scaled = scaler.fit_transform(X_train_filled)  # Learn scaling exactly once from training data.
X_test_scaled = scaler.transform(X_test_filled)  # Apply training scaling to test data.


## Compare five classifiers

KNN remains the deployed model to preserve the project's original choice. Scores from one split are exploratory, and choosing a model using these scores would require further validation.

In [ ]:
models = {  # Create all model objects before looping.
    "LR": LogisticRegression(max_iter=1000),  # Allow logistic regression to converge.
    "KNN": KNeighborsClassifier(n_neighbors=5),  # Match the deployed model.
    "NB": GaussianNB(),  # Create Naive Bayes.
    "DT": DecisionTreeClassifier(random_state=42),  # Make the tree reproducible.
    "SVM": SVC(),  # Create the support-vector classifier.
}  # Complete the model dictionary.
results = []  # Reset the list on each execution.
for name, model in models.items():  # Train each model independently.
    model.fit(X_train_scaled, Y_train)  # Fit the training subset.
    predictions = model.predict(X_test_scaled)  # Predict held-out rows.
    results.append({"Model": name, "Accuracy": accuracy_score(Y_test, predictions), "F1": f1_score(Y_test, predictions, zero_division=0)})  # Store measured scores.
pd.DataFrame(results).round(4)  # Display the comparison without discarding original score precision.


In [ ]:
knn_predictions = models["KNN"].predict(X_test_scaled)  # Inspect the deployed classifier in detail.
print(classification_report(Y_test, knn_predictions, zero_division=0))  # Display precision, recall and F1.


## Export the matching app artifacts

The shared script uses the same data, split, imputation, scaling and KNN settings. It writes the model, imputer, scaler, columns and evaluation metadata together. Restart Streamlit after retraining.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "train.py"), "--data", str(DATA_PATH)], check=True)  # Reproduce and export the complete deployment bundle.
